# Deploy DICOMweb Gateway App

Full DICOMweb server (QIDO-RS, WADO-RS, WADO-URI, STOW-RS)
backed by Databricks SQL, Lakebase, and Volumes.

In [0]:
%pip install --upgrade databricks-sdk==0.88.0 fsspec -q
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [0]:
sql_warehouse_id, table, volume = init_widgets(show_volume=True)
init_env()

In [0]:
app_gateway_name = "pixels-dicomweb-gateway"
lakebase_instance_name = "pixels-lakebase"

# Generate app.yml and Create App

In [0]:
import os

# Compute repo root from notebook path
_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicomweb_gateway_path = os.path.join(_repo_root, "apps", "dicom-web-gateway")

# Derive volume paths
_vol_parts = volume.split(".")
_stow_volume_path = f"/Volumes/{_vol_parts[0]}/{_vol_parts[1]}/{_vol_parts[2]}/stow/"

# Generate app.yml from template
with open(f"{_dicomweb_gateway_path}/app-config.yml", "r") as config_input:
    with open(f"{_dicomweb_gateway_path}/app.yml", "w") as config_output:
        config_output.write(
            config_input.read()
            .replace("{PIXELS_TABLE}", table)
            .replace("{LAKEBASE_INSTANCE_NAME}", lakebase_instance_name)
            .replace("{STOW_VOLUME_PATH}", _stow_volume_path)
        )
print(f"Generated app.yml for gateway")

In [ ]:
from databricks.sdk.service.apps import (
    AppResource,
    AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    App,
    AppDeployment,
    ComputeSize,
)

sql_resource = AppResource(
    name="sql_warehouse",
    sql_warehouse=AppResourceSqlWarehouse(
        id=sql_warehouse_id,
        permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
    ),
)

# Gateway makes OBO calls to SQL warehouses and Volumes; the proxy-injected
# x-forwarded-access-token must carry these scopes for dbsql.connect() and
# Volume reads/writes to succeed when the request originates from the viewer
# app (cross-app OBO) or directly from a user.
GATEWAY_USER_API_SCOPES = ["sql", "files.files"]

# Create or retrieve existing app
if app_gateway_name in [a.name for a in w.apps.list()]:
    print(f"App '{app_gateway_name}' already exists — updating resources")
    app = w.apps.update(
        app_gateway_name,
        App(
            app_gateway_name,
            resources=[sql_resource],
            user_api_scopes=GATEWAY_USER_API_SCOPES,
        ),
    )
else:
    print(f"Creating DICOMweb App GATEWAY '{app_gateway_name}' — this may take a few minutes …")
    app = App(
        app_gateway_name,
        default_source_code_path=_dicomweb_gateway_path,
        resources=[sql_resource],
        user_api_scopes=GATEWAY_USER_API_SCOPES,
        compute_size=ComputeSize.LARGE,
    )
    app = w.apps.create_and_wait(app)
    print(f"✓ App created: {app.url}")

# Deploy

In [0]:
app_gateway_deploy = w.apps.deploy_and_wait(app_gateway_name, AppDeployment(source_code_path=_dicomweb_gateway_path))
print(f"✅ Gateway deployed: {app_gateway_deploy.status.message}")
print(f"   URL: {w.apps.get(app_gateway_name).url}")